# Árvore de Cenários para Determinação de Propriedades de Risco para Febre Aftosa
## Convênio nº 24/2024 - IMA (Instituto Mineiro de Agropecuária)

In [ ]:
# Importação das bibliotecas
#import os
#import pandas as pd
#from pathlib import Path
#import re
#from scipy.spatial.distance import cdist
#import sys

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)# Inspecionar colunas

def checar_coluna(coluna, nome):
    print(f"\n--- {nome} ---")
    
    serie = (
        coluna.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
        .replace("nan", None)
    )
    
    print("dtype:", coluna.dtype)
    print("tem '.0'? ->", coluna.astype(str).str.contains(r"\.0$").any())
    print("distribuição de tamanho:")
    print(serie.dropna().str.len().value_counts().sort_index())

In [ ]:
from src.data_io import load_exploracao_pecuaria_raw, load_exploracoes_com_coordenadas

exploracoes_df = load_exploracao_pecuaria_raw()
exploracoes_com_coordenadas_df = load_exploracoes_com_coordenadas()

In [ ]:
print(exploracoes_df.columns)
print(exploracoes_com_coordenadas_df.columns)

In [ ]:
checar_coluna(exploracoes_com_coordenadas_df["Cód. Município"], "Município")
checar_coluna(exploracoes_com_coordenadas_df["Cód. Propriedade"], "Propriedade")
checar_coluna(exploracoes_com_coordenadas_df["CPF/CNPJ do Proprietário"], "CPF/CNPJ")
print(sys.path[:3])

In [ ]:
exploracoes_df.dtypes

In [ ]:
exploracoes_com_coordenadas_df.dtypes

In [ ]:
# Conferir se há cadastros 'nan' na coluna de CPF
print(exploracoes_df["CPF/CNPJ do Produtor"].isna().sum())

## Analisando explorações

In [ ]:
from src.preprocessing import preprocess_exploracoes

exploracoes_df = preprocess_exploracoes(exploracoes_df)

In [ ]:
exploracoes_df["CPF/CNPJ do Produtor"].str.len().value_counts(dropna=False)

In [ ]:
from src.preprocessing import clean_identifier_columns

cols_ids = [
    "Cód. Propriedade",
    "Cód. Município",
    "CPF/CNPJ do Proprietário"
]

exploracoes_com_coordenadas_df = clean_identifier_columns(
    exploracoes_com_coordenadas_df,
    cols_ids
)

In [ ]:
exploracoes_com_coordenadas_df.head()

In [ ]:
exploracoes_com_coordenadas_df.dtypes

In [ ]:
from src.debug_tools import diagnostico_digitos

In [ ]:
diagnostico_digitos(exploracoes_com_coordenadas_df, "Cód. Propriedade")
diagnostico_digitos(exploracoes_com_coordenadas_df, "Cód. Município")
diagnostico_digitos(exploracoes_com_coordenadas_df, "CPF/CNPJ do Proprietário")

In [ ]:
from src.preprocessing import clean_and_pad_identifier

exploracoes_com_coordenadas_df["Cód. Município"] = clean_and_pad_identifier(
    exploracoes_com_coordenadas_df["Cód. Município"], 7
)

exploracoes_com_coordenadas_df["Cód. Propriedade"] = clean_and_pad_identifier(
    exploracoes_com_coordenadas_df["Cód. Propriedade"], 11
)

In [ ]:
exploracoes_com_coordenadas_df.head()

In [ ]:
from src.debug_tools import checar_coluna_identificador

checar_coluna_identificador(
    exploracoes_com_coordenadas_df["Cód. Município"],
    "Município"
)

checar_coluna_identificador(
    exploracoes_com_coordenadas_df["Cód. Propriedade"],
    "Propriedade"
)

checar_coluna_identificador(
    exploracoes_com_coordenadas_df["CPF/CNPJ do Proprietário"],
    "CPF/CNPJ"
)

In [ ]:
exploracoes_com_coordenadas_df.head()

In [ ]:
from src.data_io import INTERIM_DIR

path = INTERIM_DIR / "Propriedades_Sem_Coordenadas.csv"
path.parent.mkdir(parents=True, exist_ok=True)

propriedades_sem_coordenadas_df = exploracoes_df[
    ~exploracoes_df["Código da Propriedade"].isin(
        exploracoes_com_coordenadas_df["Cód. Propriedade"]
    )
]

propriedades_sem_coordenadas_df.to_csv(path, index=False)

In [ ]:
print(len(propriedades_sem_coordenadas_df))

In [ ]:
exploracoes_com_coordenadas_df["Cód. Propriedade"] = clean_and_pad_identifier(
    exploracoes_com_coordenadas_df["Cód. Propriedade"], 11
)

exploracoes_com_coordenadas_df["Cód. Município"] = clean_and_pad_identifier(
    exploracoes_com_coordenadas_df["Cód. Município"], 7
)

In [ ]:
exploracoes_com_coordenadas_df.head()

In [ ]:
print(exploracoes_df["Código da Propriedade"].dtype)
print(exploracoes_com_coordenadas_df["Cód. Propriedade"].dtype)

In [ ]:
from src.preprocessing import filtrar_propriedades_com_coordenadas

exploracoes_df = filtrar_propriedades_com_coordenadas(
    exploracoes_df,
    exploracoes_com_coordenadas_df
)

In [ ]:
len(exploracoes_df)

In [ ]:
from src.preprocessing import adicionar_coordenadas

exploracoes_df = adicionar_coordenadas(
    exploracoes_df,
    exploracoes_com_coordenadas_df
)

In [ ]:
exploracoes_df.head()

In [ ]:
len(exploracoes_df)

In [ ]:
from src.preprocessing import adicionar_ano_atualizacao

exploracoes_df = adicionar_ano_atualizacao(exploracoes_df)


In [ ]:
exploracoes_df.head()

### Apenas explorações com bovinos

In [ ]:
limite_inferior_de_bovinos_por_exploracao = 0

In [ ]:
from src.preprocessing import filtrar_exploracoes_por_bovinos

exploracoes_bovinos_df = filtrar_exploracoes_por_bovinos(
    exploracoes_df,
    limite_inferior_de_bovinos_por_exploracao
)

In [ ]:
print(f"Redução percentual de explorações com o filtro: {100 * (1 - len(exploracoes_bovinos_df) / len(exploracoes_df)):.2f}%")

### Eliminando explorações inativas

In [ ]:
from src.preprocessing import filtrar_exploracoes_ativas

exploracoes_bovinos_df = filtrar_exploracoes_ativas(exploracoes_bovinos_df)

In [ ]:
from src.debug_tools import diagnosticar_exploracoes_bovinos

diag = diagnosticar_exploracoes_bovinos(exploracoes_bovinos_df)

diag["situacoes_exploracao"]
diag["linhas_duplicadas"]
diag["estatisticas_bovinos"]
diag["num_propriedades_distintas"]
diag["num_propriedades_mais_de_uma_exploracao"]
diag["total_exploracoes"]
diag["total_exploracoes_mais_de_uma"]

In [ ]:
diag = diagnosticar_exploracoes_bovinos(exploracoes_bovinos_df)
diag

### Dicionário de todos os bovinos por propriedade, considerando todas as explorações ativas

In [ ]:
from src.preprocessing import calcular_bovinos_por_propriedade

bovinos_por_propriedade = calcular_bovinos_por_propriedade(exploracoes_bovinos_df)

bovinos_por_propriedade.head(10)

### Considerando apenas uma exploração por propriedade

In [ ]:
from src.preprocessing import selecionar_maior_exploracao_por_propriedade

propriedades_consideradas_df = selecionar_maior_exploracao_por_propriedade(
    exploracoes_bovinos_df
)

In [ ]:
propriedades_consideradas_df.head()

In [ ]:
len(propriedades_consideradas_df)

In [ ]:
from src.preprocessing import adicionar_total_bovinos

propriedades_consideradas_df = adicionar_total_bovinos(
    propriedades_consideradas_df,
    exploracoes_bovinos_df
)

In [ ]:
propriedades_consideradas_df["Total de Bovinos"].describe()

In [ ]:
from src.debug_tools import contar_inconsistencias_bovinos

contar_inconsistencias_bovinos(propriedades_consideradas_df)

In [ ]:
from src.debug_tools import validar_total_vs_exploracao

validar_total_vs_exploracao(propriedades_consideradas_df)

In [ ]:
# reiniciar o índice de propriedades_consideradas_df
propriedades_consideradas_df = propriedades_consideradas_df.reset_index(drop=True)
propriedades_consideradas_df.head()

### Detalhamento de dados por coordenadorias

In [ ]:
from src.data_io import RAW_DIR, load_coordenadorias_dict, build_coordenadoria_mappings

coordenadorias_dict = load_coordenadorias_dict(RAW_DIR)

coord_to_esec, esec_to_coord = build_coordenadoria_mappings(coordenadorias_dict)

In [ ]:
from src.debug_tools import contar_esecs_por_coordenadoria

diag = contar_esecs_por_coordenadoria(coord_to_esec)

diag["total_esecs"]
diag["por_coordenadoria"]

In [ ]:
from src.data_io import (
    RAW_DIR,
    load_coordenadorias_dict,
    build_esec_municipio_mappings
)

coordenadorias_dict = load_coordenadorias_dict(RAW_DIR)
esec_to_mun, mun_to_esec = build_esec_municipio_mappings(coordenadorias_dict)

In [ ]:
len(esec_to_mun)          # quantas ESECs
len(mun_to_esec)          # quantos municípios

list(esec_to_mun.items())[:3]  # amostra
list(mun_to_esec.items())[:3]

In [ ]:
# selecionar todas as esecs únicas
codigos_esecs_dict = esec_to_mun.copy()

In [ ]:
from src.preprocessing import inicializar_propriedades_por_esec

propriedades_por_esec = inicializar_propriedades_por_esec(esec_to_mun)

In [ ]:
from src.preprocessing import contar_propriedades_por_esec

propriedades_por_esec = contar_propriedades_por_esec(
    propriedades_consideradas_df,
    mun_to_esec
)

In [ ]:
# criar dataframe para propriedades_por_esec
import pandas as pd

propriedades_por_esec_df = pd.DataFrame(list(propriedades_por_esec.items()), columns=['ESEC', 'Quantidade de Propriedades'])
propriedades_por_esec_df

In [ ]:
print(propriedades_por_esec_df[['Quantidade de Propriedades']].sum())
print((propriedades_por_esec_df['Quantidade de Propriedades'] == 0).sum())

# testando quantas propriedades possui CAPELINHA
print(propriedades_por_esec_df[propriedades_por_esec_df['ESEC'] == 'Capelinha'])

In [ ]:
from src.preprocessing import adicionar_esec_e_coordenadoria

propriedades_consideradas_df = adicionar_esec_e_coordenadoria(
    propriedades_consideradas_df,
    mun_to_esec,
    esec_to_coord
)

In [ ]:
propriedades_consideradas_df["ESEC"].isna().sum()

In [ ]:
from src.preprocessing import gerar_propriedades_por_esec_df
from src.data_io import RESULTS_DIR, salvar_propriedades_por_esec

propriedades_por_esec_df = gerar_propriedades_por_esec_df(propriedades_consideradas_df)

path = salvar_propriedades_por_esec(propriedades_por_esec_df, RESULTS_DIR)

print(path)

In [ ]:
from src.preprocessing import padronizar_nomes_colunas

propriedades_consideradas_df = padronizar_nomes_colunas(propriedades_consideradas_df)

In [ ]:
propriedades_consideradas_df.head()

In [ ]:
from src.data_io import INTERIM_DIR

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

path = INTERIM_DIR / "propriedades_consideradas_interim.csv"

propriedades_consideradas_df.to_csv(path, index=False)

In [ ]:
propriedades_consideradas_df.head()

In [ ]:
propriedades_consideradas_df.dtypes

## Equipe UFLA

- Mayron César de Oliveira (Departamento de Ciência da Computação - UFLA)   
    - E-mail: mayron.moreira@ufla.br   
- Sávio Augusto de Toledo Fernandes (Bolsista de IC, Engenharia Florestal - UFLA)   
- Thomas Gomes Moreira (Bolsista de IC, Engenharia Florestal - UFLA)   